In [ ]:
from pathlib import Path
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
from parameteriser import (
    plot_parameter_distribution,
    plot_parameter_distributions,
    print_organisms,
    select_organism,
    select_substrate,
)
from parameteriser.brenda.v0 import Brenda

## Use case 1: investigating distribution of enzymatic parameters


If you have downloaded the brenda database from [here](https://www.brenda-enzymes.org/download.php),you can `Brenda.read_database` to extract the needed information at once.

Otherwise, calls to `Brenda.get_kms_and_kcats` will download the data on-demand.

In [ ]:
brenda = Brenda()

if (path := Path.home() / "Documents" / "brenda_2023_1.json").exists():
    brenda.read_database(path)

You can then the saturation constant and turnover number using `Brenda.get_kms_and_kcats`.

**Note**: If the keyword argument `add_uniprot_sequences` is set to `True` (default case), additional sequence information will be download from the `uniprot` database.

In [ ]:
kms, kcats = brenda.get_kms_and_kcats(ec="4.1.1.39")
kms.head()

As there is data on quite a lot of organisms, you can use the `print_organisms` function to get an overview over the organisms, for which data exists in the specific dataframe. 

In [ ]:
print_organisms(kms, max_rows=5)

Let's first select a subset of the kms, for the substrate `CO2`.

In [ ]:
kms_co2 = select_substrate(kms, substrate="CO2")
kms_co2.head()

For which we can easily plot the entire distribution.

In [ ]:
fig, ax = plot_parameter_distribution(
    kms_co2["value"],
)
plt.show()

As enzymatic constants are different for different organisms, we further select
data for just one organisms, in this case `Nicotiana tabacum` or common tobacco.

In [ ]:
kms_tobacco_co2 = select_organism(
    select_substrate(
        kms,
        substrate="CO2",
    ),
    organism="Nicotiana tabacum",
)
kms_tobacco_co2.head()

We can now plot both distributions to compare how the data for tobacco compares with the overall data.

In [ ]:
fig, ax = plot_parameter_distributions(
    kms_co2["value"],
    kms_tobacco_co2["value"],
    organism_name="Nicotiana tabacum",
)
plt.show()

## Use case 2: selecting km based on sequence

Quite often you will have an exact protein sequence for which you would like to know the associated km and kcat values.
We provide two ways of finding these: blast and deepmolecules (**FIXME: INSERT CITATIONS**).

For illustration purposes we will take one of the brenda sequences as an example sequence:

In [ ]:
enzyme_sequence: str = kms_tobacco_co2["sequence"].iloc[0]
print(f"{enzyme_sequence[:76]}...")

### Using blast

In [ ]:
from parameteriser import blast_sequence_against_others

blast_results = blast_sequence_against_others(
    enzyme_sequence,
    kms_co2["sequence"],
)
blast_results.head()

**FIXME: GIVE GOOD DEFAULT MODE OF EVALUATING A GOOD FIT**

In [ ]:
# Just give best hit?
best_result = cast(float, kms_co2.loc[blast_results.index[0], "value"])
print(best_result)

In [ ]:
# Weighted mean of best-x?
# Or of best 95 %?

_best_ten = blast_results.iloc[:10]["pident"]
best_of_ten: float = np.average(
    kms_co2.loc[_best_ten.index, "value"],
    weights=_best_ten.values,
)
print(best_of_ten)

### Using deepmolecules

In [ ]:
from parameteriser.deepmolecules import predict_kcat, predict_km

predict_km(
    substrate="C00011",  # kegg id for CO2
    enzyme_sequence=kms_tobacco_co2["sequence"].iloc[0],
)

predict_kcat(
    substrate="C00011",  # kegg id for CO2
    product="C00197",  # kegg id for 3-Phospho-D-glycerate
    enzyme_sequence=kms_tobacco_co2["sequence"].iloc[0],
)